# Demo 1 · A locked environment

**Slides:** "Lock what ran, not what you wanted" (slide 11) and demo step 01.

Colab's preinstalled packages change without notice. Here we replace them with exactly the versions,
and the exact files (by sha256 hash), recorded in `uv.lock`.

Runtime: **CPU**. The lockfile pins the CPU build of JAX; installing it on a GPU runtime replaces Colab's CUDA build.
Takes about two minutes, plus one session restart.

## 1 · What Colab gives us today

In [ ]:
!pip list 2>/dev/null | grep -iE "^(jax|jaxlib|keras|numpy|pandas|scikit-learn|mlflow|dvc) "

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q   # already cloned in this session: get the latest version
%cd /content/repo
!git log --oneline -1

## 2 · The lockfile

`pyproject.toml` states intent (`keras>=3`). `uv.lock` records every transitive package, its exact version and the hash of every file.

In [ ]:
!grep -A3 'dependencies = \[' pyproject.toml | head -5
!grep -c '^\[\[package\]\]' uv.lock
!grep -m1 -A8 '^name = "keras"' uv.lock

## 3 · Install from the lock

`--require-hashes` makes the install fail if any downloaded file does not match its recorded hash.

In [ ]:
# Install exactly what uv.lock records, verified by sha256 hash.
!pip install -q uv
!uv export --frozen --no-emit-project --format requirements-txt -o requirements.lock.txt
!uv pip install --system -q --require-hashes -r requirements.lock.txt
!uv pip install --system -q --no-deps .
# Colab's preinstalled bottleneck is too old for the locked pandas and only causes warnings.
!uv pip uninstall --system -q bottleneck
print("Done. Now restart the session: Runtime > Restart session, then continue below.")

## 4 · After the restart: check the environment against the lock

In [ ]:
%cd /content/repo
import os, sys
os.environ["KERAS_BACKEND"] = "jax"
# Make the demo code importable here and in commands such as !dvc repro and !pytest.
os.environ["PYTHONPATH"] = "/content/repo/src"
sys.path.insert(0, "/content/repo/src")

In [ ]:
import tomllib, importlib.metadata as md

lock = tomllib.load(open("uv.lock", "rb"))
locked = {}
for pkg in lock["package"]:
    locked.setdefault(pkg["name"], set()).add(pkg.get("version"))

checked, mismatched = 0, []
for name, versions in sorted(locked.items()):
    try:
        installed = md.version(name)
    except md.PackageNotFoundError:
        continue  # platform-specific package not needed here
    checked += 1
    if installed not in versions and name != "repro-demo":
        mismatched.append((name, sorted(v for v in versions if v), installed))

print(f"{checked} packages checked against uv.lock")
for name, want, got in mismatched:
    print(f"  MISMATCH {name}: locked {want}, installed {got}")
print("environment matches the lock" if not mismatched else "")

## 5 · Record the hardware too

The lock covers Python packages. Devices, drivers and the runtime image are logged with every run (see demo 2).

In [ ]:
import jax, platform
print("python", platform.python_version(), "| jax", jax.__version__, "|", jax.devices())
!nvidia-smi --query-gpu=name,driver_version --format=csv 2>/dev/null || echo "no GPU on this runtime"